# Download NASA POWER Solar Irradiance Data for Toronto

**Purpose:** Download solar irradiance data for Toronto coordinates to enhance electricity demand forecasting.

**Data Source:** NASA POWER (Prediction of Worldwide Energy Resources)
- API: https://power.larc.nasa.gov/api/

**Location:** Toronto, Ontario
- Latitude: 43.65°N
- Longitude: -79.38°W

**Parameters:**
- **ALLSKY_SFC_SW_DWN**: Global Horizontal Irradiance (GHI) - total solar radiation on horizontal surface
- **ALLSKY_SFC_SW_DNI**: Direct Normal Irradiance (DNI) - direct beam radiation
- **ALLSKY_SFC_SW_DIFF**: Diffuse Horizontal Irradiance (DHI) - scattered radiation

**Time Range:** June 2013 - November 2025

**Why this matters:** Solar irradiance affects electricity demand through:
1. Solar PV generation (reduces net demand from grid)
2. Cloud cover patterns (affects lighting loads)
3. Seasonal daylight variations (commercial demand timing)

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
import time
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [2]:
# Toronto coordinates
latitude = 43.65
longitude = -79.38

# Date range (matching your IESO data: June 2013 - November 2025)
start_date = "20130601"
end_date = "20251130"

# Solar parameters to download
# GHI = Global Horizontal Irradiance (total solar on horizontal surface)
# DNI = Direct Normal Irradiance (direct beam radiation)
# DHI = Diffuse Horizontal Irradiance (scattered radiation)
parameters = "ALLSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF"

# Build the API URL
base_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
api_url = (
    f"{base_url}?"
    f"parameters={parameters}&"
    f"community=RE&"  # RE = Renewable Energy community
    f"longitude={longitude}&"
    f"latitude={latitude}&"
    f"start={start_date}&"
    f"end={end_date}&"
    f"format=JSON"
)

print("NASA POWER API Request:")
print(f"  Location: {latitude}°N, {longitude}°W (Toronto)")
print(f"  Date range: {start_date} to {end_date}")
print(f"  Parameters: GHI, DNI, DHI")
print(f"\nAPI URL:")
print(api_url)

NASA POWER API Request:
  Location: 43.65°N, -79.38°W (Toronto)
  Date range: 20130601 to 20251130
  Parameters: GHI, DNI, DHI

API URL:
https://power.larc.nasa.gov/api/temporal/daily/point?parameters=ALLSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF&community=RE&longitude=-79.38&latitude=43.65&start=20130601&end=20251130&format=JSON


In [3]:
# Make the API request
print("Downloading solar data from NASA POWER...")
print("(This may take 30-60 seconds...)\n")

try:
    response = requests.get(api_url, timeout=120)
    
    # Check if request was successful
    if response.status_code == 200:
        print("✓ API request successful!")
        
        # Parse JSON response
        data = response.json()
        
        # Check response structure
        print(f"\nResponse keys: {list(data.keys())}")
        
        # Extract the parameter data
        if 'properties' in data and 'parameter' in data['properties']:
            solar_data = data['properties']['parameter']
            print(f"\nAvailable parameters: {list(solar_data.keys())}")
            
            # Check how many days of data we got
            ghi_data = solar_data['ALLSKY_SFC_SW_DWN']
            print(f"\nNumber of days downloaded: {len(ghi_data)}")
            print(f"First date: {list(ghi_data.keys())[0]}")
            print(f"Last date: {list(ghi_data.keys())[-1]}")
            
        else:
            print("Unexpected response structure")
            print(data)
    else:
        print(f"✗ API request failed with status code: {response.status_code}")
        print(f"Response: {response.text}")
        
except Exception as e:
    print(f"✗ Error occurred: {str(e)}")

(This may take 30-60 seconds...)

✓ API request successful!

Response keys: ['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times']

Available parameters: ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF']

Number of days downloaded: 4552
First date: 20130601
Last date: 20251116


In [4]:
# Extract the solar parameter data
ghi_data = solar_data['ALLSKY_SFC_SW_DWN']  # Global Horizontal Irradiance
dni_data = solar_data['ALLSKY_SFC_SW_DNI']  # Direct Normal Irradiance  
dhi_data = solar_data['ALLSKY_SFC_SW_DIFF'] # Diffuse Horizontal Irradiance

# Convert to DataFrame
df_solar = pd.DataFrame({
    'Date': list(ghi_data.keys()),
    'GHI': list(ghi_data.values()),
    'DNI': list(dni_data.values()),
    'DHI': list(dhi_data.values())
})

# Convert date strings to datetime
df_solar['Date'] = pd.to_datetime(df_solar['Date'], format='%Y%m%d')

# Convert solar values to numeric (handle any missing values marked as -999)
for col in ['GHI', 'DNI', 'DHI']:
    df_solar[col] = pd.to_numeric(df_solar[col], errors='coerce')
    # Replace -999 (NASA's missing value code) with NaN
    df_solar.loc[df_solar[col] == -999, col] = np.nan

print("Solar data converted to DataFrame")
print(f"\nShape: {df_solar.shape}")
print(f"Date range: {df_solar['Date'].min()} to {df_solar['Date'].max()}")
print(f"\nMissing values:")
print(df_solar.isna().sum())
print(f"\nFirst 10 rows:")
df_solar.head(10)

Solar data converted to DataFrame

Shape: (4552, 4)
Date range: 2013-06-01 00:00:00 to 2025-11-16 00:00:00

Missing values:
Date      0
GHI       7
DNI     109
DHI     109
dtype: int64

First 10 rows:


,Date,GHI,DNI,DHI
0,2013-06-01,4.4316,1.2190,3.3902
1,2013-06-02,5.2046,3.4334,2.6774
2,2013-06-03,8.4106,10.4393,1.3315
3,2013-06-04,8.3244,10.2214,1.2096
4,2013-06-05,5.8639,3.2083,3.4584
5,2013-06-06,1.8389,0.1121,1.4796
6,2013-06-07,2.6239,0.5954,1.9723
7,2013-06-08,3.9038,1.8098,2.6426
8,2013-06-09,7.6366,7.3901,2.2606
9,2013-06-10,1.9200,0.0758,1.5502


In [5]:
# Basic statistics
print("Solar Irradiance Statistics (kWh/m²/day):")
print(df_solar[['GHI', 'DNI', 'DHI']].describe())

# Check for any unusual values
print(f"\nValue ranges:")
print(f"GHI: {df_solar['GHI'].min():.2f} to {df_solar['GHI'].max():.2f} kWh/m²/day")
print(f"DNI: {df_solar['DNI'].min():.2f} to {df_solar['DNI'].max():.2f} kWh/m²/day")
print(f"DHI: {df_solar['DHI'].min():.2f} to {df_solar['DHI'].max():.2f} kWh/m²/day")

# Show last few rows to verify end date
print(f"\nLast 5 rows:")
df_solar.tail()

Solar Irradiance Statistics (kWh/m²/day):
               GHI          DNI          DHI
count  4545.000000  4443.000000  4443.000000
mean      3.711941     3.578492     1.641338
std       2.213664     2.877013     0.809727
min       0.207600     0.000000     0.175700
25%       1.726600     0.871200     0.933400
50%       3.412800     3.100800     1.537400
75%       5.544000     5.780800     2.241100
max       8.870900    11.688200     4.491400

Value ranges:
GHI: 0.21 to 8.87 kWh/m²/day
DNI: 0.00 to 11.69 kWh/m²/day
DHI: 0.18 to 4.49 kWh/m²/day

Last 5 rows:


,Date,GHI,DNI,DHI
4547,2025-11-12,NaN,NaN,NaN
4548,2025-11-13,NaN,NaN,NaN
4549,2025-11-14,NaN,NaN,NaN
4550,2025-11-15,NaN,NaN,NaN
4551,2025-11-16,NaN,NaN,NaN


In [6]:
# Save the solar data to CSV
output_path = "../../02_Datasets/processed/nasa_solar_2013_2025.csv"
df_solar.to_csv(output_path, index=False)

print(f"✓ Solar data saved to: {output_path}")
print(f"\nFinal solar dataset:")
print(f"  Shape: {df_solar.shape}")
print(f"  Date range: {df_solar['Date'].min()} to {df_solar['Date'].max()}")
print(f"  Columns: {df_solar.columns.tolist()}")
print(f"  Missing values: {df_solar.isna().sum().sum()} total")
print(f"  Memory usage: {df_solar.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n{'='*70}")
print(f"NASA POWER SOLAR DATA DOWNLOAD COMPLETE! ✅")
print(f"{'='*70}")

✓ Solar data saved to: ../../02_Datasets/processed/nasa_solar_2013_2025.csv

Final solar dataset:
  Shape: (4552, 4)
  Date range: 2013-06-01 00:00:00 to 2025-11-16 00:00:00
  Columns: ['Date', 'GHI', 'DNI', 'DHI']
  Missing values: 225 total
  Memory usage: 0.14 MB

NASA POWER SOLAR DATA DOWNLOAD COMPLETE! ✅
